In [1]:
import numpy as np 
import math
import pandas as pd 

In [6]:
data = pd.read_csv('CIC2018.csv')
# test = pd.read_csv('UNSW_NB15_testing-set.csv')
# combined_data = pd.concat([train, test]).drop(['id'],axis=1)


D:\Anaconda3\envs\TF_36aa\lib\site-packages\IPython\core\interactiveshell.py:2698: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [25]:
combined_data = data

In [26]:
from sklearn.preprocessing import LabelEncoder,normalize
le1 = LabelEncoder()
le = LabelEncoder()

vector = combined_data['Label']
print("Label:", set(list(vector))) # use print to make it print on single line 

combined_data['Label'] = le1.fit_transform(vector)
# combined_data['proto'] = le.fit_transform(combined_data['proto'])
# combined_data['service'] = le.fit_transform(combined_data['service'])
# combined_data['state'] = le.fit_transform(combined_data['state'])

vector = combined_data['Label']

Label: {0, 1, 2}


In [27]:
combined_data = data[~data['Label'].isin([2])] 

In [28]:
y_label = combined_data.loc[:,['Label']].values.flatten()
dict = {}
for i in y_label:
    dict.update({i:dict.get(i,0)+1})
dict

{0: 238037, 1: 93063}

In [30]:
le1.inverse_transform([0,1])
combined_data.head(3)

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0,0,01/03/2018 08:17:11,115307855,5,0,0,0,0,0,...,0,1812348,0,1812348,1812348,56700000,6010057.622,61000000,52500000,0
1,0,0,01/03/2018 08:20:07,60997457,2,0,0,0,0,0,...,0,0,0,0,0,61000000,0,61000000,61000000,0
2,67,17,01/03/2018 08:17:18,61149019,5,0,1500,0,300,300,...,8,3530939,0,3530939,3530939,19200000,12500000,32600000,7999725,0


In [34]:
# ## OMITTED: For statistical feature removal

# lowSTD = list(combined_data.std().to_frame().nsmallest(6, columns=0).index)
# # this is stupid. suppose a feature has a 1.0 (spearman or pearson) correlation, OR conditional probability, when not 0.... That a very useful feature  

# lowCORR = list(combined_data.corr().abs().sort_values('Label')['Label'].nsmallest(3).index) # .where(lambda x: x < 0.005).dropna()
# # This might be stupid. A Deep MLP (feed forward neural net) may see patterns

# drop = set( lowCORR + lowSTD)
# # drop = {'ackdat', 'ct_ftp_cmd', 'djit', 'is_ftp_login', 'is_sm_ips_ports', 'response_body_len', 'sjit', 'synack', 'tcprtt'}
# # print(f'Before {combined_data.shape}')
# combined_data_reduced=combined_data.drop(drop,axis=1)
# # print(f'After {combined_data.shape}')

In [40]:
combined_data_reduced=combined_data.drop('Timestamp',axis=1)

## 无穷值处理
combined_data_reduced.replace(np.inf, 0, inplace=True)

In [41]:
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import math
import matplotlib.pyplot as plt

# import seaborn as sns
import sklearn.metrics as metrics
%matplotlib inline
import os
from pandas_ml import ConfusionMatrix

In [42]:
data_x = combined_data_reduced.drop(['Label'], axis=1) # droped label
data_y = combined_data_reduced.loc[:,['Label']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO

y_train = y_train.values.flatten()
y_test = y_test.values.flatten()